<a href="https://colab.research.google.com/github/faisu6339-glitch/Deep-Learning/blob/main/RNN_BPTT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Backpropagation Through Time (BPTT)

Backpropagation Through Time (BPTT) is an extension of the standard backpropagation algorithm used to train recurrent neural networks (RNNs). Unlike feedforward neural networks, RNNs have connections that loop back on themselves, allowing them to maintain an internal state and process sequences of data. BPTT handles these recurrent connections by treating the RNN as an unrolled feedforward network over time.

### Why Do We Need BPTT?

In a normal neural network, information flows in one direction:

Input → Hidden → Output

We use standard Backpropagation to update weights. However, in an RNN, the same weights are reused at every time step to process sequential data:

x₁ → h₁
       ↓
x₂ → h₂
       ↓
x₃ → h₃
       ↓
      Output

Since the weights (`W_xh`, `W_hh`, `W_hy`) are shared across all time steps, ordinary backpropagation is insufficient. We need an algorithm that can account for the influence of these shared weights across the entire sequence, which is precisely what Backpropagation Through Time (BPTT) does by propagating errors backward across time steps.

### Intuition

Imagine an RNN processing a sequence of inputs, say words in a sentence. At each time step, the RNN receives an input and also considers its internal state (hidden state) from the previous time step. It then produces an output and updates its internal state. This process repeats for the entire sequence.

To train this RNN, we need to calculate how much each weight contributes to the overall error (loss) at the end of the sequence. This is where BPTT comes in. Instead of just backpropagating through a single layer, BPTT "unrolls" the RNN over all time steps, effectively creating a very deep feedforward network.

### How it Works (Mathematical Intuition)

1.  **Unrolling the Network:** The first step is to visualize or conceptually unroll the recurrent neural network. If an RNN processes a sequence of length `T`, you can think of it as `T` copies of the same network, where the hidden state of one copy feeds into the next.

    *   At each time step `t`, the hidden state `h_t` is typically computed as:
        `h_t = tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)`
    *   The output `y_t` at time `t` is then computed as:
        `y_t = W_{hy} h_t + b_y`

    Here, `x_t` is the input at time `t`, `h_t` is the hidden state, `W_{xh}` are input-to-hidden weights, `W_{hh}` are recurrent weights, `W_{hy}` are hidden-to-output weights, and `b_h`, `b_y` are bias vectors. Crucially, ***these weights (`W_{xh}`, `W_{hh}`, `W_{hy}`) are shared across all time steps***.

    **Unrolled RNN Visualization:**

    ```
t=1        t=2        t=3

x₁ ──► h₁ ──► h₂ ──► h₃
        │       │       │
        ▼       ▼       ▼
       y₁      y₂      y₃
    ```

    This diagram illustrates that the same `W_xh`, `W_hh`, and `W_hy` are used at every time step.

2.  **Forward Pass:** The unrolled network is run forward from `t=1` to `t=T`, computing hidden states (`h_1, h_2, ..., h_T`) and outputs (`y_1, y_2, ..., y_T`) for each time step.

3.  **Calculate Loss:** A loss function `L` is calculated, typically as the sum of losses at each time step, comparing predicted outputs `y_t` with target outputs `ŷ_t`:

    `L_t = (1/2) * (y_t - ŷ_t)^2` (for mean squared error)

    The total loss is the sum over all time steps:

    `L = Σ_{t=1}^{T} L_t`

    For a 3-step sequence: `L = L_1 + L_2 + L_3`

4.  **Understanding Dependency:**

    The core challenge is that the loss at a given time step `t` depends on the current hidden state `h_t`, which in turn depends on all previous hidden states (`h_{t-1}, h_{t-2}, ..., h_0`) and all previous inputs (`x_t, x_{t-1}, ..., x_1`).

    For example, `h_3 = f(x_3, h_2)`, but `h_2 = f(x_2, h_1)`, and `h_1 = f(x_1, h_0)`. Therefore, `h_3` contains information derived from `x_1, x_2, x_3` and `h_0`. This means the error at time `T` (or any `L_t`) must flow backward through all previous states to correctly update the shared weights.

5.  **Backward Pass (Backpropagation):** This is where the "Through Time" part becomes vital. We need to compute the gradients of the total loss `L` with respect to all shared parameters (`W_{xh}`, `W_{hh}`, `W_{hy}`, `b_h`, `b_y`).

    *   **Chain Rule in BPTT:** For any parameter `W` (e.g., `W_{hh}`), its gradient `∂L/∂W` is calculated by summing up the gradients from each time step `t`:

        `∂L/∂W = Σ_t (∂L_t/∂W)`

        More specifically, for a recurrent weight like `W_{hh}`, the gradient calculation involves tracing the dependency through time:

        `∂L/∂W_{hh} = Σ_{t=1}^{T} (∂L/∂h_t) * (∂h_t/∂W_{hh})`

        The term `∂L/∂h_t` itself is recursive. The backpropagation process starts at the last time step `T` and moves backward to `t=1`. At each time step `t`, the gradient `∂L/∂h_t` (the gradient of the total loss with respect to the hidden state at time `t`) is calculated. This gradient has two components:
        1.  The direct influence of `h_t` on the loss at time `t` (`L_t`), i.e., `∂L_t/∂h_t`.
        2.  The influence of `h_t` on the hidden state at time `t+1` (`h_{t+1}`), which in turn affects the loss from `t+1` onwards. This part is `(∂L/∂h_{t+1}) * (∂h_{t+1}/∂h_t)`.

        So, the gradient `∂L/∂h_t` is given by:

        `∂L/∂h_t = ∂L_t/∂h_t + (∂L/∂h_{t+1}) * (∂h_{t+1}/∂h_t)`

        (with `∂L/∂h_{T+1}` usually defined as 0).

        Once `∂L/∂h_t` is computed for all `t`, the gradients for the weights and biases are calculated using the standard chain rule, summing contributions from each time step.

    *   **Derivative of Hidden State (Example for tanh activation):**

        If `h_t = tanh(a_t)` where `a_t = W_{xh} x_t + W_{hh} h_{t-1} + b_h`,

        Then `∂h_t/∂a_t = 1 - tanh^2(a_t) = 1 - h_t^2`

        And `∂a_t/∂h_{t-1} = W_{hh}`

        Therefore, `∂h_t/∂h_{t-1} = (∂h_t/∂a_t) * (∂a_t/∂h_{t-1}) = W_{hh} (1 - h_t^2)`

    *   **Long Gradient Chain:** The crucial part is that the influence of an earlier hidden state `h_k` on a later hidden state `h_T` (and thus `L_T`) involves a product of these derivatives:

        `∂h_T/∂h_k = Π_{i=k+1}^{T} (∂h_i/∂h_{i-1}) = Π_{i=k+1}^{T} [W_{hh} (1 - h_i^2)]`

        This repeated multiplication is at the heart of BPTT's challenges.

### Challenges: Vanishing and Exploding Gradients

The long chains of multiplications in the gradient calculations can lead to:

*   **Vanishing Gradients:** If `|W_{hh} (1 - h_i^2)| < 1` (e.g., `W_{hh} = 0.5` and `(1 - h_i^2)` is also less than 1), the product `Π [W_{hh} (1 - h_i^2)]` becomes exponentially smaller as `T-k` increases. For example, `0.5^10 = 0.000976` and `0.5^50 ≈ 8.88 × 10^-16`. This means gradients from far-away time steps effectively disappear, making it difficult for the network to learn long-term dependencies (early words cannot influence the final output significantly).

    **Intuition:**
    Error at output
          ↓
    h₅₀
          ↓
    h₄₉
          ↓
    ...
          ↓
    h₁
    Each step back multiplies by a small number, eventually making the gradient for `h_1` almost zero.

*   **Exploding Gradients:** Conversely, if `|W_{hh} (1 - h_i^2)| > 1` (e.g., `W_{hh} = 2`), the product can become exponentially large. For example, `2^20 = 1,048,576`. This leads to extremely large gradients, causing unstable training, large weight updates, and often model divergence.

    **Effect:**
    Loss
     ↓
    Huge gradient
     ↓
    Weight update explodes
     ↓
    Training unstable

### Gradient Descent Weight Update

After computing the gradients, weights are updated using an optimization algorithm like Gradient Descent:

`W_{new} = W_{old} - η * (∂L/∂W)`

Where `η` is the learning rate.

### Complete BPTT Algorithm (Summary)

1.  **Forward Pass:** For each time step `t` from 1 to `T`:
    *   Compute hidden state `h_t` based on `x_t` and `h_{t-1}`.
    *   Compute output `y_t` based on `h_t`.
    *   Calculate loss `L_t`.
2.  **Backward Pass:** For each time step `t` from `T` down to 1:
    *   Compute `∂L/∂h_t` by backpropagating through `L_t` and `h_{t+1}`.
    *   Compute gradients `∂L/∂W_{xh}`, `∂L/∂W_{hh}`, `∂L/∂W_{hy}` (and biases) using `∂L/∂h_t`.
    *   Accumulate these gradients over all time steps.
3.  **Weight Update:** Update the shared weights and biases using the accumulated gradients and a learning rate.

### Computational Complexity

For a sequence of length `T`:

*   **Memory Complexity:** `O(T)` because all hidden states (`h_1, ..., h_T`) need to be stored during the forward pass for use in the backward pass.
*   **Time Complexity:** `O(T)` per sequence, as computations are performed for each time step in both forward and backward passes.

For very long sequences, this can become computationally expensive and memory-intensive.

### Truncated BPTT

To mitigate the high memory and computational cost of full BPTT for very long sequences, **Truncated Backpropagation Through Time (TBPTT)** is often used. Instead of backpropagating through *all* previous steps (e.g., `h_1 ← ... ← h_T`), TBPTT limits the backpropagation to a fixed number of recent time steps (e.g., `h_{T-k} ← ... ← h_T`). This significantly reduces memory usage and computation per update, although it might make it harder to learn extremely long-term dependencies.

### Why LSTM and GRU Were Invented

Standard RNNs trained with BPTT suffer significantly from:

*   Vanishing gradients, making it hard to capture long-term dependencies.
*   Exploding gradients, leading to training instability.

Long Short-Term Memory (LSTM) and Gated Recurrent Unit (GRU) networks were specifically designed to address these problems. They introduce internal 'gates' (Forget, Input, Output gates in LSTMs) and a 'cell state' that provide a nearly constant error flow path, allowing gradients to propagate more effectively over long sequences and thereby greatly reducing the vanishing-gradient problem.

### Exam Definition

**Backpropagation Through Time (BPTT)** is an extension of the backpropagation algorithm used for recurrent neural networks, in which the network is conceptually unrolled through time. The total loss is computed over all time steps, and gradients are propagated backward through each hidden state, layer, and time step using the chain rule. This process accumulates gradients for the shared weights (`W_{xh}`, `W_{hh}`, `W_{hy}`) and biases, enabling the RNN to learn temporal dependencies in sequential data, despite the challenges posed by vanishing and exploding gradients.

### Visual Summary: Vanishing Gradient Problem

Imagine the flow of gradients during backpropagation through time. The further back in time we go, the smaller the influence of errors becomes on earlier weights.

```
Error at Output (L_T)
    ↓ (Large Gradient)
    h_T (Last Hidden State)
    ↓ (Gradient Multiplied by small factor < 1)
    h_{T-1}
    ↓ (Gradient Multiplied by small factor < 1)
    h_{T-2}
    ↓ (...
    ↓ (...
    ↓ (Gradient Multiplied by small factor < 1)
    h_2
    ↓ (Gradient Multiplied by small factor < 1)
    h_1 (First Hidden State)

Result:
Gradient for W_hh and W_xh associated with h_1 ≈ 0

```

This visual represents how the `∂h_i/∂h_{i-1}` terms, when repeatedly multiplied and being less than 1, shrink the error signal to negligible values for earlier time steps, effectively 'forgetting' long-term dependencies.

### BPTT vs. Truncated BPTT: Complexity Comparison

When training Recurrent Neural Networks (RNNs), Backpropagation Through Time (BPTT) and Truncated BPTT (TBPTT) are used to compute gradients. Their complexities differ significantly, especially for long sequences.

#### Full Backpropagation Through Time (BPTT)

*   **Concept:** Backpropagates errors through *all* time steps of a sequence.
*   **Computational Complexity:**
    *   **Time:** `O(T * N)` where `T` is the sequence length and `N` is the complexity of a single time step's forward/backward pass. This means the time to compute gradients is proportional to the entire sequence length.
    *   **Implication:** For very long sequences, this can be extremely slow, as the forward and backward passes have to traverse the entire unrolled network.
*   **Memory Complexity:**
    *   `O(T * D)` where `T` is the sequence length and `D` is the dimensionality of the hidden state. This is because all intermediate hidden states (`h_1, ..., h_T`) must be stored during the forward pass to be used in the backward pass for gradient calculation.
    *   **Implication:** Can quickly exhaust memory for long sequences, making it impractical for tasks like processing long documents or audio.
*   **Gradient Accuracy:** Provides the most accurate gradients for the entire sequence, capturing long-range dependencies perfectly (though still subject to vanishing/exploding gradients).

#### Truncated Backpropagation Through Time (TBPTT)

*   **Concept:** Limits the backpropagation of errors to a fixed number of recent time steps (a `k`-step window), rather than the entire sequence.
*   **Computational Complexity:**
    *   **Time:** `O(k * N)` where `k` is the truncation length (the number of steps to backpropagate) and `N` is the complexity of a single time step's forward/backward pass. The time complexity becomes independent of the total sequence length `T`, and depends only on `k`.
    *   **Implication:** Significantly faster for very long sequences (`T >> k`) as computations are localized within the truncation window.
*   **Memory Complexity:**
    *   `O(k * D)` where `k` is the truncation length and `D` is the dimensionality of the hidden state. Only the hidden states within the `k`-step window need to be stored.
    *   **Implication:** Much more memory-efficient than full BPTT, allowing training on longer sequences that wouldn't fit into memory otherwise.
*   **Gradient Accuracy:** Provides an approximation of the gradients. It might struggle to learn very long-term dependencies that span beyond the `k`-step truncation window, as error signals from earlier time steps are not propagated all the way back.

#### Summary Table

| Feature             | Full BPTT               | Truncated BPTT (TBPTT)      |
| :------------------ | :---------------------- | :-------------------------- |
| **Time Complexity** | `O(T * N)`              | `O(k * N)` (where `k << T`) |
| **Memory Complexity** | `O(T * D)`              | `O(k * D)` (where `k << T`) |
| **Gradient Accuracy** | Exact (within numerical limits) | Approximate                 |
| **Long-Term Dependencies** | Theoretically perfect (if no vanishing/exploding) | Limited by truncation length `k` |
| **Practicality**    | Impractical for very long sequences | Widely used for long sequences |

In essence, TBPTT sacrifices some gradient accuracy and ability to capture extremely long-range dependencies for practical gains in computational speed and memory efficiency, making it the preferred method for many real-world RNN applications.

### BPTT vs. Truncated BPTT: Key Differences for a Slide Deck

#### Full Backpropagation Through Time (BPTT)

*   **Scope:** Backpropagates errors through the *entire* sequence.
*   **Complexity:**
    *   **Time:** `O(T)` - Proportional to the full sequence length.
    *   **Memory:** `O(T)` - Stores all hidden states for the full sequence.
*   **Gradient Accuracy:** Exact gradients for the entire sequence.
*   **Long-Term Dependencies:** Theoretically captures all dependencies (but limited by vanishing/exploding gradients).
*   **Practicality:** Impractical for very long sequences due to high resource demands.

#### Truncated Backpropagation Through Time (TBPTT)

*   **Scope:** Backpropagates errors through a *fixed window* (`k`) of recent time steps.
*   **Complexity:**
    *   **Time:** `O(k)` - Proportional to the fixed window size `k` (independent of total sequence length `T`).
    *   **Memory:** `O(k)` - Stores hidden states only within the `k`-step window.
*   **Gradient Accuracy:** Approximates gradients; loses information from beyond the window.
*   **Long-Term Dependencies:** Limited to the `k`-step window.
*   **Practicality:** Widely used and essential for training on long sequences due to efficiency.